# Oil Market Analytical Report (WTI)

This report analyzes the historical dynamics of West Texas Intermediate (WTI) oil prices using the dataset available in the project. It focuses on trend, volatility, and anomaly detection, then builds a forecasting model to provide a disciplined view of potential future price behavior.

**Scope:** Annual WTI price series (from project data) with supplementary production context.
**Objective:** Deliver a professional, decision-ready analytical narrative with transparent methodology.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.figsize": (11, 5.5),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 10
})

data_dir = Path("..") / "data"
prices = pd.read_csv(data_dir / "oil_prices.csv")
production = pd.read_csv(data_dir / "oil_production.csv")

prices.head()

## 1. Executive Summary

We quantify the overall price regime, long-run growth, and the most material drawdowns. These metrics anchor the rest of the analysis and provide a compact business-ready view of the market's behavior.

In [ ]:
prices_clean = prices.copy()
prices_clean["year"] = pd.to_datetime(prices_clean["year"].astype(str) + "-12-31")
prices_clean = prices_clean.sort_values("year").reset_index(drop=True)

prices_clean["yoy_pct_change"] = prices_clean["oil_price"].pct_change() * 100
prices_clean["drawdown"] = prices_clean["oil_price"] / prices_clean["oil_price"].cummax() - 1

start_price = prices_clean["oil_price"].iloc[0]
end_price = prices_clean["oil_price"].iloc[-1]
years = (prices_clean["year"].iloc[-1] - prices_clean["year"].iloc[0]).days / 365.25
cagr = (end_price / start_price) ** (1 / years) - 1
worst_drawdown = prices_clean["drawdown"].min()
max_yoy = prices_clean["yoy_pct_change"].max()
min_yoy = prices_clean["yoy_pct_change"].min()

summary = pd.DataFrame({
    "Metric": ["Start price", "End price", "CAGR", "Worst drawdown", "Max YoY change", "Min YoY change"],
    "Value": [
        f"${start_price:,.0f}",
        f"${end_price:,.0f}",
        f"{cagr * 100:,.2f}%",
        f"{worst_drawdown * 100:,.1f}%",
        f"{max_yoy:,.1f}%",
        f"{min_yoy:,.1f}%"
    ]
})
summary

## 2. Data Cleaning

We standardize date fields, remove duplicates, validate numeric types, and confirm the absence of gaps. This ensures downstream computations are deterministic and trustworthy.

In [ ]:
prices_clean = prices_clean.drop_duplicates(subset=["year"])
prices_clean["oil_price"] = pd.to_numeric(prices_clean["oil_price"], errors="coerce")

missing = prices_clean.isna().sum().to_frame(name="missing")
prices_clean = prices_clean.dropna().reset_index(drop=True)

missing

## 3. Exploratory Data Analysis

The EDA section quantifies baseline price behavior, highlights distributional properties, and anchors context with a snapshot of major oil producers.

In [ ]:
display(prices_clean.describe(include="all"))

fig, ax = plt.subplots()
ax.plot(prices_clean["year"], prices_clean["oil_price"], color="#1f4e79", linewidth=2)
ax.set_title("WTI Price Trend (Annual)")
ax.set_xlabel("Year")
ax.set_ylabel("Price (USD)")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

top_producers = production.sort_values("production", ascending=False).head(8)
fig, ax = plt.subplots()
ax.barh(top_producers["country"], top_producers["production"], color="#6b8ba4")
ax.invert_yaxis()
ax.set_title("Top Oil Producing Countries (Snapshot)")
ax.set_xlabel("Production (kbpd)")
plt.tight_layout()
plt.show()

## 4. Trend Analysis

To assess short- and medium-term trend dynamics, we upsample the annual series to daily frequency using linear interpolation. This enables 7-day and 30-day rolling mean estimates for consistent smoothing.

In [ ]:
daily = prices_clean.set_index("year")["oil_price"].asfreq("D")
daily = daily.interpolate(method="time")

daily_df = daily.to_frame(name="oil_price")
daily_df["rolling_7d"] = daily_df["oil_price"].rolling(window=7, min_periods=3).mean()
daily_df["rolling_30d"] = daily_df["oil_price"].rolling(window=30, min_periods=10).mean()

fig, ax = plt.subplots()
ax.plot(daily_df.index, daily_df["oil_price"], color="#c0c0c0", linewidth=1, label="Daily (interp)")
ax.plot(daily_df.index, daily_df["rolling_7d"], color="#1f4e79", linewidth=2, label="7D Rolling Mean")
ax.plot(daily_df.index, daily_df["rolling_30d"], color="#f28e2b", linewidth=2, label="30D Rolling Mean")
ax.set_title("WTI Rolling Mean Trends")
ax.set_xlabel("Date")
ax.set_ylabel("Price (USD)")
ax.legend(frameon=False)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 5. Volatility Analysis

Volatility is measured using rolling standard deviation of daily returns. This highlights regimes of price stress and market instability.

In [ ]:
daily_df["pct_change"] = daily_df["oil_price"].pct_change()
daily_df["rolling_vol_30d"] = daily_df["pct_change"].rolling(window=30, min_periods=10).std() * np.sqrt(30)

fig, ax = plt.subplots()
ax.plot(daily_df.index, daily_df["rolling_vol_30d"], color="#7a4e2d", linewidth=2)
ax.set_title("30-Day Rolling Volatility (Annualized)")
ax.set_xlabel("Date")
ax.set_ylabel("Volatility")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 6. Supply Shock Detection

We flag abnormal price moves using a z-score on daily returns. These anomalies are interpreted as potential supply shocks or demand disruptions, warranting further investigation.

In [ ]:
returns = daily_df["pct_change"].dropna()
z_scores = (returns - returns.mean()) / returns.std()
anomalies = z_scores.abs() > 2.5

shock_df = pd.DataFrame({
    "date": returns.index,
    "return": returns.values,
    "z_score": z_scores.values,
    "shock": anomalies.values
}).query("shock").sort_values("z_score")

fig, ax = plt.subplots()
ax.plot(daily_df.index, daily_df["pct_change"], color="#9aa0a6", linewidth=1)
ax.scatter(returns.index[anomalies], returns[anomalies], color="#c00000", s=30, label="Shock")
ax.set_title("Return Shocks (Z-Score > 2.5)")
ax.set_xlabel("Date")
ax.set_ylabel("Daily Return")
ax.legend(frameon=False)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

shock_df.head(10)

## 7. Feature Engineering

We construct lagged values, rolling statistics, and return-based features. These engineered features serve as inputs to predictive models and improve explanatory power.

In [ ]:
feature_df = daily_df.copy()
feature_df["lag_1"] = feature_df["oil_price"].shift(1)
feature_df["lag_7"] = feature_df["oil_price"].shift(7)
feature_df["lag_30"] = feature_df["oil_price"].shift(30)
feature_df["rolling_mean_7"] = feature_df["oil_price"].rolling(window=7).mean()
feature_df["rolling_mean_30"] = feature_df["oil_price"].rolling(window=30).mean()
feature_df["rolling_std_30"] = feature_df["oil_price"].rolling(window=30).std()
feature_df["pct_change"] = feature_df["oil_price"].pct_change()

feature_df = feature_df.dropna()
feature_df.head()

## 8. Machine Learning Forecasting

An ARIMA model is fitted on the annual WTI series to capture persistent trend and mean-reversion components. This provides a transparent statistical baseline for forecasting.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

annual_series = prices_clean.set_index("year")["oil_price"]
train = annual_series.iloc[:-2]
test = annual_series.iloc[-2:]

model = ARIMA(train, order=(1, 1, 1))
model_fit = model.fit()
forecast = model_fit.get_forecast(steps=len(test))
forecast_df = forecast.summary_frame()
forecast_df.index = test.index

forecast_df.head()

## 9. Prediction Visualization

The forecast is compared against historical prices with confidence bounds for risk-aware interpretation.

In [ ]:
fig, ax = plt.subplots()
ax.plot(annual_series.index, annual_series.values, color="#1f4e79", linewidth=2, label="Historical")
ax.plot(test.index, forecast_df["mean"], color="#f28e2b", linewidth=2, label="Forecast")
ax.fill_between(
    test.index,
    forecast_df["mean_ci_lower"],
    forecast_df["mean_ci_upper"],
    color="#f28e2b", alpha=0.2, label="Confidence Interval"
)
ax.set_title("ARIMA Forecast vs Historical WTI Prices")
ax.set_xlabel("Year")
ax.set_ylabel("Price (USD)")
ax.legend(frameon=False)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 10. Risk and Limitations

- The core price series is annual, which reduces sensitivity to intra-year shocks and short-term volatility.
- Daily interpolation is used strictly for rolling-window analytics and should not be interpreted as real daily observations.
- ARIMA provides a statistical baseline, not a causal model, and does not account for policy, geopolitics, or structural breaks.

## 11. Final Insights

- WTI prices exhibit clear regime shifts with pronounced drawdowns and rapid recoveries.
- Rolling trend and volatility measures isolate periods of heightened market stress and price instability.
- A simple ARIMA baseline supports directional forecasting but should be complemented with macro and inventory indicators for investment-grade decisions.